In [175]:
# import
import glob
import pandas as pd

In [176]:
# 1. 파일 읽기

# data/baemin 폴더에서 baemin으로 시작하는 csv 파일 경로를 모두 찾아 리스트로 돌려줌
# * 는 "아무 글자나"라는 뜻 → baemin20261001.csv, baemin20261002.csv ... 전부 찾음
# sorted() 로 날짜 순서대로 정렬
# 파일 목록은 아래 힌트 코드로 가져오고
baemin_files = sorted(glob.glob("data/baemin/baemin*.csv"))
len(baemin_files)

coupang_files = sorted(glob.glob("data/coupang/coupang*.csv"))
len(coupang_files)

31

In [177]:
#

#  baemin 폴더의 파일 31개를 모두 읽어서 하나의 DataFrame으로 만드세요. 
# 반복문으로 하나씩 읽어서 합치세요.
baemin_list = []
for add in baemin_files:
    df = pd.read_csv(add)
    baemin_list.append(df)

baemin = pd.concat(baemin_list)
len(baemin)


1155

In [178]:
# coupang 폴더의 파일 31개도 같은 방식으로 읽으세요. 
# 반복문으로 하나씩 읽어서 합치세요.
coupang_list = []
for add in coupang_files:
    df = pd.read_csv(add)
    coupang_list.append(df)

coupang = pd.concat(coupang_list)
len(coupang)

712

In [179]:
# 2. 칼럼명 통일

# 두 플랫폼의 칼럼명을 주문일시, 전체금액, 광고유입여부로 바꾸세요.
baemin = baemin.rename(columns={
    "order_at":"주문일시",
    "total_bill":"전체금액",
    "ad_yn":"광고유입여부"
})

# 어느 플랫폼 데이터인지 알 수 있도록 플랫폼 칼럼(배민 / 쿠팡)을 추가하세요.
baemin["플랫폼"] = "배민"
# 두 플랫폼 모두 주문 1건마다 광고집행비(원) 칼럼을 만드세요.(배민: cpc 그대로)
baemin["광고집행비"] = baemin["cpc"]
baemin

,주문일시,전체금액,광고유입여부,cpc,플랫폼,광고집행비
0,2026-10-01 11:48:34,26000,Y,700,배민,700
1,2026-10-01 11:53:22,15000,N,0,배민,0
2,2026-10-01 12:18:52,35000,N,0,배민,0
3,2026-10-01 12:28:44,35000,N,0,배민,0
4,2026-10-01 12:50:50,35000,N,0,배민,0
...,...,...,...,...,...,...
36,2026-10-31 20:34:53,17000,N,0,배민,0
37,2026-10-31 20:47:07,38000,Y,640,배민,640
38,2026-10-31 21:39:35,19000,Y,690,배민,690
39,2026-10-31 21:45:29,45000,N,0,배민,0


In [180]:
# 두 플랫폼의 칼럼명을 주문일시, 전체금액, 광고유입여부로 바꾸세요.
coupang = coupang.rename(columns={
    "order_time":"주문일시",
    "total_price":"전체금액",
    "advertisement_yn":"광고유입여부"
})

# 어느 플랫폼 데이터인지 알 수 있도록 플랫폼 칼럼(배민 / 쿠팡)을 추가하세요.
coupang["플랫폼"] = "쿠팡"
# 두 플랫폼 모두 주문 1건마다 광고집행비(원) 칼럼을 만드세요.(쿠팡: 전체금액 × ad_pct / 100)
coupang["광고집행비"] = coupang["전체금액"] * coupang["ad_pct"] / 100
coupang

,주문일시,전체금액,광고유입여부,ad_pct,플랫폼,광고집행비
0,2026-10-01 11:46:07,18000,Y,10,쿠팡,1800.0
1,2026-10-01 12:06:50,21000,Y,10,쿠팡,2100.0
2,2026-10-01 12:16:29,58000,Y,10,쿠팡,5800.0
3,2026-10-01 12:19:49,43000,Y,10,쿠팡,4300.0
4,2026-10-01 13:00:05,58000,N,0,쿠팡,0.0
...,...,...,...,...,...,...
22,2026-10-31 19:33:36,32000,N,0,쿠팡,0.0
23,2026-10-31 19:36:08,18000,Y,5,쿠팡,900.0
24,2026-10-31 19:50:22,18000,N,0,쿠팡,0.0
25,2026-10-31 19:59:04,34000,N,0,쿠팡,0.0


In [181]:
# 3. 데이터 통합

# 두 DataFrame을 하나로 합치세요. (pd.concat)
col = ["주문일시", "전체금액", "광고유입여부", "플랫폼", "광고집행비"]
order = pd.concat([baemin[col], coupang[col]])
# order

# 주문일시를 datetime으로 바꾸고, 
order["주문일시"] = pd.to_datetime(order["주문일시"])
# order["주문일시"]

# 날짜만 담은 주문일자 칼럼을 추가하세요.
order["주문일자"] = order["주문일시"].dt.date
# order["주문일자"] 

# 주문일시 순으로 정렬하세요.
order = order.sort_values("주문일시")
order = order.reset_index(drop=True)
order


,주문일시,전체금액,광고유입여부,플랫폼,광고집행비,주문일자
0,2026-10-01 11:46:07,18000,Y,쿠팡,1800.0,2026-10-01
1,2026-10-01 11:48:34,26000,Y,배민,700.0,2026-10-01
2,2026-10-01 11:53:22,15000,N,배민,0.0,2026-10-01
3,2026-10-01 12:06:50,21000,Y,쿠팡,2100.0,2026-10-01
4,2026-10-01 12:16:29,58000,Y,쿠팡,5800.0,2026-10-01
...,...,...,...,...,...,...
1862,2026-10-31 20:47:07,38000,Y,배민,640.0,2026-10-31
1863,2026-10-31 20:49:23,24000,N,쿠팡,0.0,2026-10-31
1864,2026-10-31 21:39:35,19000,Y,배민,690.0,2026-10-31
1865,2026-10-31 21:45:29,45000,N,배민,0.0,2026-10-31


In [182]:
# 4. 일자별 매출 집계

# 행은 주문일자, 열은 플랫폼, 값은 전체금액 합계인 표를 만드세요. (pivot_table)
daily_order = order.pivot_table(
    index="주문일자",
    columns="플랫폼",
    values="전체금액",
    aggfunc="sum"
)
daily_order

# 합계 열을 추가하세요.
daily_order["합계"] = daily_order["배민"] + daily_order["쿠팡"]
daily_order

플랫폼,배민,쿠팡,합계
주문일자,,,
2026-10-01,959000,562000,1521000
2026-10-02,1348000,904000,2252000
2026-10-03,1294000,779000,2073000
2026-10-04,1127000,928000,2055000
2026-10-05,1103000,543000,1646000
2026-10-06,1028000,576000,1604000
2026-10-07,1227000,549000,1776000
2026-10-08,946000,439000,1385000
2026-10-09,1200000,835000,2035000


In [183]:
# 5. 광고 효과 분석
mask = (order["광고유입여부"] == "Y")
ad = order[mask]

# 플랫폼별로 광고유입 주문이 전체 주문 건수에서 차지하는 비율(%)을 구하세요.
total = order.groupby("플랫폼").size()
ad_total = ad.groupby("플랫폼").size()

ad_per_order = ad_total / total * 100
print("광고 주문 비율:\n", round(ad_per_order, 2))

# 플랫폼별 광고집행비 합계를 구하세요.
ad_sum = order.groupby("플랫폼")["광고집행비"].sum()
print("\n플랫폼별 광고집행비:\n", ad_sum)


광고 주문 비율:
 플랫폼
배민    34.98
쿠팡    51.97
dtype: float64

플랫폼별 광고집행비:
 플랫폼
배민    243170.0
쿠팡    886700.0
Name: 광고집행비, dtype: float64


In [184]:
# 6. 식자재비 추출
# 파일이 안열려 오류를 검색해 보니 encoding='cp949'을 해보라해서 적용했습니다.
bank = pd.read_csv("data/bank_20261001_20261031.csv", encoding='cp949')
bank

# 은행 거래내역에서 거래처가 정확히 `식자재마트`인 출금 내역만 골라내세요.
bank_mask = bank["거래처"] == "식자재마트"
bank_result = bank[bank_mask]
bank_result


,거래일시,적요,거래처,출금액,입금액,잔액
1,2026.10.01 09:01,체크카드,식자재마트,197100,0,4796800
5,2026.10.02 09:15,체크카드,식자재마트,267100,0,6193900
7,2026.10.02 15:03,체크카드,식자재마트,113300,0,6056000
10,2026.10.03 09:07,체크카드,식자재마트,194800,0,5437000
11,2026.10.03 15:28,체크카드,식자재마트,52800,0,5384200
13,2026.10.04 09:08,체크카드,식자재마트,289300,0,5082700
15,2026.10.05 09:44,체크카드,식자재마트,214400,0,4859900
20,2026.10.06 09:04,체크카드,식자재마트,209500,0,6969700
24,2026.10.07 09:03,체크카드,식자재마트,273100,0,6540300
26,2026.10.07 15:16,체크카드,식자재마트,59700,0,6470700


In [185]:
# 일자별 식자재비 합계를 구하세요.
bank_result["거래일시"] = pd.to_datetime(bank_result["거래일시"])
daily_bank = bank_result.set_index("거래일시")
daily_bank = daily_bank.resample("D")["출금액"].sum()
daily_bank.name = "식자재비"
daily_bank


거래일시
2026-10-01    197100
2026-10-02    380400
2026-10-03    247600
2026-10-04    289300
2026-10-05    214400
2026-10-06    209500
2026-10-07    332800
2026-10-08    219300
2026-10-09    351300
2026-10-10    324900
2026-10-11    225000
2026-10-12    305500
2026-10-13    314100
2026-10-14    375200
2026-10-15    288500
2026-10-16    389700
2026-10-17    342000
2026-10-18    184600
2026-10-19    250100
2026-10-20    205600
2026-10-21    292400
2026-10-22    263500
2026-10-23    289100
2026-10-24    289800
2026-10-25    312300
2026-10-26    272800
2026-10-27    200400
2026-10-28    391000
2026-10-29    213300
2026-10-30    302800
2026-10-31    397300
Freq: D, Name: 식자재비, dtype: int64

In [186]:
# 7. 최종 결과표 저장

# 4번 결과와 6번 결과를 일자 기준으로 합쳐서 아래 칼럼을 가진 표를 만드세요.
csv_result = daily_order.join(daily_bank)
csv_result

# 매출-식자재비
csv_result["매출-식자재비"] = csv_result["합계"] - csv_result["식자재비"]
csv_result

csv_result = csv_result.reset_index()
csv_result.columns.name = None
csv_result


C:\Users\kstae\AppData\Local\Temp\ipykernel_32680\347937436.py:4: Pandas4Warning: Indexing a DatetimeIndex with a sequence of datetime.date objects is deprecated. Use Timestamp objects instead.
  csv_result = daily_order.join(daily_bank)


,주문일자,배민,쿠팡,합계,식자재비,매출-식자재비
0,2026-10-01,959000,562000,1521000,197100,1323900
1,2026-10-02,1348000,904000,2252000,380400,1871600
2,2026-10-03,1294000,779000,2073000,247600,1825400
3,2026-10-04,1127000,928000,2055000,289300,1765700
4,2026-10-05,1103000,543000,1646000,214400,1431600
5,2026-10-06,1028000,576000,1604000,209500,1394500
6,2026-10-07,1227000,549000,1776000,332800,1443200
7,2026-10-08,946000,439000,1385000,219300,1165700
8,2026-10-09,1200000,835000,2035000,351300,1683700
9,2026-10-10,1379000,719000,2098000,324900,1773100


In [187]:
# 결과를 result.csv로 저장하세요. (엑셀에서 열어도 한글이 안 깨지게 encoding="utf-8-sig")
csv_result.to_csv("result.csv", index=False, encoding="utf-8-sig")